In [1]:
tabla='qtsoo10'
col_tabla = 'solopefec'
essi='essi_dat_cqx007_etl'
col_essi='sol_fec'

In [2]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [5]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"
# c1= text(query)
# connection2.execute(c1)

#INICIO DEL ESSI

In [6]:
fecha = '01/05/23'

In [7]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.SOLOPEORICENASICOD,  
d1.SOLOPECENASICOD,  
d1.SOLOPENUM,
d1.SOLOPECPSCOD,
d1.SOLOPEFLGPRI,
d1.SOLOPEPQXCPSCOD,
d1.SOLOPEPQXSECUEN,
d1.SOLOPESOLMATDOCNUM,
d1.SOLOPEGRDCOMOPECOD,a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,  
a1.SOLOPEBUSPACSECNUM
from {tabla} d1  
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.SOLOPEORICENASICOD  
and a1.SOLOPECENASICOD = d1.SOLOPECENASICOD  
and a1.SOLOPENUM = d1.SOLOPENUM
where a1.{col_tabla}>='{fecha}'

"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [8]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100216 entries, 0 to 100215
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   solopeoricenasicod  100216 non-null  object        
 1   solopecenasicod     100216 non-null  object        
 2   solopenum           100216 non-null  int64         
 3   solopecpscod        100216 non-null  object        
 4   solopeflgpri        65458 non-null   float64       
 5   solopepqxcpscod     65458 non-null   object        
 6   solopepqxsecuen     65458 non-null   float64       
 7   solopesolmatdocnum  65458 non-null   float64       
 8   solopegrdcomopecod  21365 non-null   object        
 9   solopefec           100216 non-null  datetime64[ns]
 10  solopeactmednum     100216 non-null  int64         
 11  solopebuspacsecnum  100216 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(5)
memory usage: 9.2+ MB


In [9]:
base1.columns

Index(['solopeoricenasicod', 'solopecenasicod', 'solopenum', 'solopecpscod',
       'solopeflgpri', 'solopepqxcpscod', 'solopepqxsecuen',
       'solopesolmatdocnum', 'solopegrdcomopecod', 'solopefec',
       'solopeactmednum', 'solopebuspacsecnum'],
      dtype='object')

In [10]:
base1 = base1.rename(columns={
    'solopeoricenasicod': 'ori_cas',
    'solopecenasicod': 'cod_cas',
    'solopenum': 'sol_num',
    'solopecpscod': 'cod_cps',
    'solopeflgpri': 'flg_pri',
    'solopepqxcpscod': 'paq_qui',
    'solopepqxsecuen': 'paq_sec',
    'solopesolmatdocnum': 'sol_med',
    'solopegrdcomopecod': 'grd_com',
    'solopefec': 'sol_fec',
    'solopeactmednum': 'act_med',
    'solopebuspacsecnum': 'pac_sec'
})

In [11]:
base1.shape

(100216, 12)

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [13]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ori_cas  0 non-null      object
 1   cod_cas  0 non-null      object
 2   des_cas  0 non-null      object
 3   cod_red  0 non-null      object
 4   des_red  0 non-null      object
 5   sol_num  0 non-null      object
 6   cod_cps  0 non-null      object
 7   flg_pri  0 non-null      object
 8   paq_qui  0 non-null      object
 9   paq_sec  0 non-null      object
 10  sol_med  0 non-null      object
 11  grd_com  0 non-null      object
 12  sol_fec  0 non-null      object
 13  act_med  0 non-null      object
 14  pac_sec  0 non-null      object
dtypes: object(15)
memory usage: 0.0+ bytes


#TRAEMOS TODOS LOS MAESTROS

In [14]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red


In [15]:
a=base1.copy()

In [16]:
# base1=a

In [17]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100216 entries, 0 to 100215
Data columns (total 12 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   ori_cas  100216 non-null  object        
 1   cod_cas  100216 non-null  object        
 2   sol_num  100216 non-null  int64         
 3   cod_cps  100216 non-null  object        
 4   flg_pri  65458 non-null   float64       
 5   paq_qui  65458 non-null   object        
 6   paq_sec  65458 non-null   float64       
 7   sol_med  65458 non-null   float64       
 8   grd_com  21365 non-null   object        
 9   sol_fec  100216 non-null  datetime64[ns]
 10  act_med  100216 non-null  int64         
 11  pac_sec  100216 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(5)
memory usage: 9.2+ MB


In [18]:
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(100216, 15)

In [19]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 100216 entries, 0 to 100215
Data columns (total 15 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   ori_cas  100216 non-null  object        
 1   cod_cas  100216 non-null  object        
 2   sol_num  100216 non-null  int64         
 3   cod_cps  100216 non-null  object        
 4   flg_pri  65458 non-null   float64       
 5   paq_qui  65458 non-null   object        
 6   paq_sec  65458 non-null   float64       
 7   sol_med  65458 non-null   float64       
 8   grd_com  21365 non-null   object        
 9   sol_fec  100216 non-null  datetime64[ns]
 10  act_med  100216 non-null  int64         
 11  pac_sec  100216 non-null  int64         
 12  des_cas  100216 non-null  object        
 13  cod_red  100216 non-null  object        
 14  des_red  100216 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(3), object(8)
memory usage: 12.2+ MB


In [20]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'sol_num',
       'cod_cps', 'flg_pri', 'paq_qui', 'paq_sec', 'sol_med', 'grd_com',
       'sol_fec', 'act_med', 'pac_sec'],
      dtype='object')

In [21]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [22]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [23]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [24]:
base.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 100216 entries, 0 to 100215
Data columns (total 15 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   sol_med  65458 non-null   float64       
 1   flg_pri  65458 non-null   float64       
 2   cod_cps  100216 non-null  object        
 3   sol_num  100216 non-null  int64         
 4   sol_fec  100216 non-null  datetime64[ns]
 5   paq_sec  65458 non-null   float64       
 6   act_med  100216 non-null  int64         
 7   cod_red  100216 non-null  object        
 8   des_red  100216 non-null  object        
 9   pac_sec  100216 non-null  int64         
 10  cod_cas  100216 non-null  object        
 11  des_cas  100216 non-null  object        
 12  ori_cas  100216 non-null  object        
 13  paq_qui  65458 non-null   object        
 14  grd_com  21365 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(3), object(8)
memory usage: 12.2+ MB


In [25]:
base.head()

,sol_med,flg_pri,cod_cps,sol_num,sol_fec,paq_sec,act_med,cod_red,des_red,pac_sec,cod_cas,des_cas,ori_cas,paq_qui,grd_com
0,0.0,1.0,44970,107872,2023-05-01,0.0,13384739,07,RED PRESTACIONAL REBAGLIATI ...,8481822,001,H.N. EDGARDO REBAGLIATI MARTINS,1,44970,None
1,0.0,1.0,31600,107858,2023-05-01,0.0,13250473,07,RED PRESTACIONAL REBAGLIATI ...,14378568,001,H.N. EDGARDO REBAGLIATI MARTINS,1,31600,None
2,0.0,1.0,43752,107860,2023-05-01,0.0,13379565,07,RED PRESTACIONAL REBAGLIATI ...,15369381,001,H.N. EDGARDO REBAGLIATI MARTINS,1,43752,None
3,NaN,NaN,44970,107866,2023-05-01,NaN,13384132,07,RED PRESTACIONAL REBAGLIATI ...,20441151,001,H.N. EDGARDO REBAGLIATI MARTINS,1,None,None
4,0.0,1.0,67973,107869,2023-05-01,0.0,13383731,07,RED PRESTACIONAL REBAGLIATI ...,27996548,001,H.N. EDGARDO REBAGLIATI MARTINS,1,67973,None


In [26]:
base.columns

Index(['sol_med', 'flg_pri', 'cod_cps', 'sol_num', 'sol_fec', 'paq_sec',
       'act_med', 'cod_red', 'des_red', 'pac_sec', 'cod_cas', 'des_cas',
       'ori_cas', 'paq_qui', 'grd_com'],
      dtype='object')

In [32]:
base['grd_com'].unique()

array([None, 'A', 'C', 'E', 'B', ' ', 'D'], dtype=object)

In [27]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [28]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

10216

In [29]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 19.293 segundos


In [30]:
connection1.close()
connection2.close()
connection3.close()

In [31]:
engine1.dispose()
engine2.dispose()
engine3.dispose()